# Baseline MLE-STAR × MLE-bench Lite (backbone pareado)

Este notebook roda o **grafo oficial do MLE-STAR** com um adaptador fino de provider (implementação
open-source do Google em [google/adk-samples](https://github.com/google/adk-samples), sample
`machine-learning-engineering`)
nas **22 competições configuradas do MLE-bench Lite** (as que excedem o limite de disco são
reportadas como puladas), na **mesma infra Colab** e com o **mesmo backbone**
Gemini 3.0 Flash Preview (`google/gemini-3-flash-preview` via OpenRouter) usado pelo Kaggle Agents.

**Por que isso existe:** a comparação da tese usa números transplantados do paper do MLE-STAR
(Gemini 2.0 Flash / 2.5 Pro, cluster 8×V100, 3 seeds). Re-rodar o MLE-STAR com o mesmo modelo e
mesma máquina elimina o confound de backbone/infra e produz uma comparação de custo *medida* —
os dois pontos mais frágeis para publicação.

**Requisitos**
1. `OPENROUTER_API_KEY` — em Colab, salve em *Secrets* (ícone de chave).
2. Secrets `KAGGLE_USERNAME` e `KAGGLE_KEY`.
3. **Aceite as regras de cada competição no site do Kaggle** (uma vez por conta), senão o
   `mlebench prepare` falha com 403.
4. GPU (L4 recomendada) e disco: competições >10 GB podem estourar o disco do Colab
   (o MLE-STAR copia os dados para cada branch de solução). Veja `SKIP_LARGE_GB` abaixo.

**Saídas:** `RESULTS_DIR/results_mlestar.json` (por competição: medalhas, above_median, score bruto,
tempo de execução, seed), `runtime_manifest.json`, `pip_freeze.txt` e tabela comparativa final no formato da
Table 5 da monografia.

**Referências:** Nam et al., *MLE-STAR: Machine Learning Engineering Agent via Search and Targeted
Refinement* (NeurIPS 2025, arXiv:2506.15692); Chan et al., *MLE-bench* (arXiv:2410.07095).

In [ ]:
# 1) Dependências
# torch/pandas/sklearn já vêm no Colab; instalamos o ADK, o grader do MLE-bench e libs
# que as soluções geradas costumam importar.
ADK_SAMPLES_COMMIT = "68989de5a041a0be2321bfdb9f7d657b148e0558"
MLEBENCH_COMMIT = "507f92e1138bb6e40dac5c6ee7a6758e6424bf97"
!pip -q install "google-adk[extensions]==1.36.1" "litellm==1.83.14" python-dotenv kaggle lightgbm xgboost catboost
# O git clone anônimo do GitHub sofre throttling intermitente no Colab (exit 128);
# o tarball do commit via codeload é mais leve e confiável. Tentativas: tarball ×2, git ×1.
import shutil
import subprocess
import sys
import time

_mlebench_sources = [
    f"https://github.com/openai/mle-bench/archive/{MLEBENCH_COMMIT}.tar.gz",
    f"https://github.com/openai/mle-bench/archive/{MLEBENCH_COMMIT}.tar.gz",
    f"git+https://github.com/openai/mle-bench.git@{MLEBENCH_COMMIT}",
]
for _attempt, _src in enumerate(_mlebench_sources, 1):
    if shutil.which("mlebench"):
        break
    print(f"Instalando mle-bench (tentativa {_attempt}/{len(_mlebench_sources)})...")
    _proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", _src])
    if _proc.returncode == 0 and shutil.which("mlebench"):
        break
    time.sleep(20)
!test -d /content/adk-samples/.git || git clone https://github.com/google/adk-samples.git /content/adk-samples
!git -C /content/adk-samples fetch --depth 1 origin {ADK_SAMPLES_COMMIT}
!git -C /content/adk-samples checkout --detach {ADK_SAMPLES_COMMIT}

# Verificação: uma sessão rodou o loop inteiro sem o CLI do mle-bench instalado
# (12 competições queimadas com FileNotFoundError em segundos).
import shutil

assert shutil.which("mlebench"), (
    "CLI `mlebench` não instalado — o pip acima falhou. Re-execute esta célula "
    "e leia o erro do pip (remova o -q da linha do mle-bench se necessário)."
)
print("Instalação OK: mlebench em", shutil.which("mlebench"))

In [ ]:
# 2) Credenciais
import os
from pathlib import Path


def _load_secret(name: str) -> str | None:
    """Read a secret from Colab userdata or the environment."""
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)


# As chaves ficam apenas em variáveis Python do orquestrador, nunca no ambiente herdado
# pelos scripts gerados.
openrouter_key = _load_secret("OPENROUTER_API_KEY")
assert openrouter_key, "Defina OPENROUTER_API_KEY (Colab Secrets ou env)"

# Kaggle (para mlebench prepare)
kaggle_user = _load_secret("KAGGLE_USERNAME")
kaggle_key = _load_secret("KAGGLE_KEY")
assert kaggle_user and kaggle_key, (
    "Defina KAGGLE_USERNAME e KAGGLE_KEY (Colab Secrets ou env)"
)
for secret_name in (
    "OPENROUTER_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY",
    "OPENAI_API_KEY", "GOOGLE_API_KEY",
):
    os.environ.pop(secret_name, None)
print("Credenciais OK")

In [ ]:
# 3) Configuração do experimento
import hashlib

# Backbone pareado com o Kaggle Agents (tese), usando o slug operacional do OpenRouter.
MODEL = "google/gemini-3-flash-preview"
LITELLM_MODEL = f"openrouter/{MODEL}"
SEARCH_MODEL = LITELLM_MODEL
SEARCH_TOOL = "openrouter:web_search"
RUN_PROVIDER_SMOKE_TEST = True  # testa modelo e busca antes da execução cara

# Protocolo. Fase atual: 1 seed (paridade com a run única da tese);
# expandir para [42, 43, 44] na run final do paper (protocolo de 3 seeds do MLE-STAR).
SEEDS = [42]

# Orçamentos. Protocolo congelado em 7h/competição (cabe na sessão do Colab);
# desvio das 24h do paper do MLE-STAR — divulgar na seção de experimentos.
MAX_WALL_CLOCK_S = 7 * 3600   # limite soft; subprocessos síncronos podem ultrapassá-lo
EXEC_TIMEOUT_S = 3600         # timeout por script gerado (config.exec_timeout)
MAX_LLM_CALLS = 4000          # teto anti-runaway do ADK (default 500 truncou runs em ~4h;
                              # o orçamento do protocolo é o wall-clock, não este teto)

# Forma do grafo do MLE-STAR (defaults do sample ADK; valores do paper em comentário)
NUM_SOLUTIONS = 2             # paper: 2
NUM_MODEL_CANDIDATES = 4      # paper: 4
OUTER_LOOP_ROUND = 4          # paper: 4
INNER_LOOP_ROUND = 4          # paper: 4
ENSEMBLE_LOOP_ROUND = 5       # paper: 5
USE_DATA_LEAKAGE_CHECKER = True
USE_DATA_USAGE_CHECKER = True

# Competições >SKIP_LARGE_GB são puladas (o MLE-STAR copia os dados por branch de solução;
# siim-isic ~116 GB não cabe no disco padrão do Colab). None = tentar todas.
SKIP_LARGE_GB = 20.0

# Resultados persistem no Drive; o workspace e o cache continuam no disco rápido local.
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
        RESULTS_DIR = Path("/content/drive/MyDrive/kaggle-agents/mlestar_results")
    except ImportError:
        print("Google Drive indisponível; usando /content (não persistente).")
        RESULTS_DIR = Path("/content/mlestar_results")
else:
    RESULTS_DIR = Path("/content/mlestar_results")
WORK_ROOT = Path("/content/mlestar_work")
MLEBENCH_DATA_DIR = Path.home() / ".cache" / "mle-bench" / "data"
CLEANUP_AFTER_RUN = True      # limpa workspace por seed e dados após todas as seeds
FORCE_RERUN = False           # True re-executa competições já presentes no results_mlestar.json
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# MLE-bench Lite: 22 competições (task_type/lower alimentam os prompts do MLE-STAR)
COMPETITIONS = [
    {"id": "aerial-cactus-identification",                     "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 0.025},
    {"id": "aptos2019-blindness-detection",                    "task_type": "Image Classification", "metric": "quadratic_weighted_kappa",  "lower": False, "size_gb": 10.22},
    {"id": "dog-breed-identification",                         "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.75},
    {"id": "dogs-vs-cats-redux-kernels-edition",               "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.85},
    {"id": "histopathologic-cancer-detection",                 "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 7.76},
    {"id": "leaf-classification",                              "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.036},
    {"id": "plant-pathology-2020-fgvc7",                       "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 0.8},
    {"id": "ranzcr-clip-catheter-line-classification",         "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 13.13},
    {"id": "siim-isic-melanoma-classification",                "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 116.16},
    {"id": "denoising-dirty-documents",                        "task_type": "Image to Image",       "metric": "rmse",                      "lower": True,  "size_gb": 0.06},
    {"id": "detecting-insults-in-social-commentary",           "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.002},
    {"id": "jigsaw-toxic-comment-classification-challenge",    "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.06},
    {"id": "random-acts-of-pizza",                             "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.003},
    {"id": "spooky-author-identification",                     "task_type": "Text Classification",  "metric": "log_loss",                  "lower": True,  "size_gb": 0.002},
    {"id": "new-york-city-taxi-fare-prediction",               "task_type": "Tabular Regression",   "metric": "rmse",                      "lower": True,  "size_gb": 5.7},
    {"id": "nomad2018-predict-transparent-conductors",         "task_type": "Tabular Regression",   "metric": "rmsle",                     "lower": True,  "size_gb": 0.006},
    {"id": "tabular-playground-series-dec-2021",               "task_type": "Tabular Classification", "metric": "accuracy",                "lower": False, "size_gb": 0.7},
    {"id": "tabular-playground-series-may-2022",               "task_type": "Tabular Classification", "metric": "auc",                     "lower": False, "size_gb": 0.57},
    {"id": "mlsp-2013-birds",                                  "task_type": "Audio Classification", "metric": "auc",                       "lower": False, "size_gb": 0.585},
    {"id": "the-icml-2013-whale-challenge-right-whale-redux",  "task_type": "Audio Classification", "metric": "auc",                       "lower": False, "size_gb": 0.29},
    {"id": "text-normalization-challenge-english-language",    "task_type": "Sequence to Sequence", "metric": "accuracy",                  "lower": False, "size_gb": 0.01},
    {"id": "text-normalization-challenge-russian-language",    "task_type": "Sequence to Sequence", "metric": "accuracy",                  "lower": False, "size_gb": 0.01},
]
competition_signature = hashlib.sha256(
    ",".join(comp["id"] for comp in COMPETITIONS).encode()
).hexdigest()[:12]
disk_cap = "all" if SKIP_LARGE_GB is None else str(SKIP_LARGE_GB)
RUN_FINGERPRINT = (
    f"openrouter:{MODEL}:{ADK_SAMPLES_COMMIT[:12]}:{MLEBENCH_COMMIT[:12]}:"
    f"c{competition_signature}-s{'-'.join(map(str, SEEDS))}-"
    f"n{NUM_SOLUTIONS}-m{NUM_MODEL_CANDIDATES}-o{OUTER_LOOP_ROUND}-"
    f"i{INNER_LOOP_ROUND}-e{ENSEMBLE_LOOP_ROUND}-"
    f"leak{int(USE_DATA_LEAKAGE_CHECKER)}-usage{int(USE_DATA_USAGE_CHECKER)}-"
    f"exec{EXEC_TIMEOUT_S}-wall{MAX_WALL_CLOCK_S}-cap{disk_cap}"
)
print(f"{len(COMPETITIONS)} competições configuradas | modelo: {MODEL} | seeds: {SEEDS}")

In [ ]:
# 4) Importar o MLE-STAR (ORDEM IMPORTA)
# O grafo é construído no import. Por isso configuramos o objeto LiteLlm e substituímos
# a busca nativa do ADK antes de importar qualquer módulo do sample.
import glob
import importlib
import importlib.machinery
import importlib.metadata
import json
import platform
import subprocess
import sys
import time
import types as py_types

agent_py = glob.glob("/content/adk-samples/**/machine_learning_engineering/agent.py", recursive=True)
assert agent_py, "Sample machine-learning-engineering não encontrado no adk-samples"
SAMPLE_DIR = Path(agent_py[0]).parent.parent  # .../machine-learning-engineering
PACKAGE_DIR = SAMPLE_DIR / "machine_learning_engineering"
if str(SAMPLE_DIR) not in sys.path:
    sys.path.insert(0, str(SAMPLE_DIR))
print(f"MLE-STAR sample: {SAMPLE_DIR}")

# Patches mínimos de bugs do sample oficial, aplicados no clone ANTES do import
# (documentados nas notas metodológicas e no runtime_manifest):
# (1) check_leakage_util registra replace_leakage_code sem functools.partial(prefix=...)
#     -> TypeError quando o leakage checker (ligado no protocolo do paper) acha leakage;
# (2) common_util concatena Part.text, que no google-genai pode ser None (resposta
#     vazia/filtrada, frequente via OpenRouter) -> TypeError nos fluxos de debug.
_leakage_path = PACKAGE_DIR / "shared_libraries" / "check_leakage_util.py"
_leakage_src = _leakage_path.read_text(encoding="utf-8")
_leakage_fixed = _leakage_src.replace(
    "        after_model_callback=replace_leakage_code,",
    "        after_model_callback=functools.partial(\n"
    "            replace_leakage_code,\n"
    "            prefix=prefix,\n"
    "        ),",
)
if _leakage_fixed != _leakage_src:
    _leakage_path.write_text(_leakage_fixed, encoding="utf-8")
    print("Patch upstream (1): replace_leakage_code com prefix")
assert "replace_leakage_code,\n            prefix=prefix" in _leakage_path.read_text(
    encoding="utf-8"
), "patch (1) não aplicou — commit do adk-samples mudou?"

_common_path = PACKAGE_DIR / "shared_libraries" / "common_util.py"
_common_src = _common_path.read_text(encoding="utf-8")
_common_fixed = _common_src.replace(
    '            if hasattr(response.content.parts[i], "text"):',
    '            if getattr(response.content.parts[i], "text", None):',
)
if _common_fixed != _common_src:
    _common_path.write_text(_common_fixed, encoding="utf-8")
    print("Patch upstream (2): Part.text None-safe em get_text_from_response")
assert 'getattr(response.content.parts[i], "text", None)' in _common_path.read_text(
    encoding="utf-8"
), "patch (2) não aplicou — commit do adk-samples mudou?"

# (3) O front-door do sample seeda os campos do CONFIG no session state; com o
#     backbone trocado, CONFIG.agent_model é um objeto LiteLlm (upstream é uma
#     string). O callback final save_state faz json.dump do estado inteiro e
#     morre com TypeError DEPOIS de o pipeline completar — default=str preserva
#     o dump sem alterar comportamento para valores já serializáveis.
_agent_path = PACKAGE_DIR / "agent.py"
_agent_src = _agent_path.read_text(encoding="utf-8")
_agent_fixed = _agent_src.replace(
    "        json.dump(callback_context.state.to_dict(), f, indent=2)",
    "        json.dump(callback_context.state.to_dict(), f, indent=2, default=str)",
)
if _agent_fixed != _agent_src:
    _agent_path.write_text(_agent_fixed, encoding="utf-8")
    print("Patch upstream (3): save_state JSON com default=str")
assert "indent=2, default=str" in _agent_path.read_text(encoding="utf-8"), (
    "patch (3) não aplicou — commit do adk-samples mudou?"
)

# (4) update_extract_status: resposta vazia/erro do provedor (finish_reason
#     'error' via OpenRouter) pula o if e o parse levanta exceção sem atribuir
#     leakage_status -> UnboundLocalError ao gravar no estado. Default neutro
#     antes do try; o parse sobrescreve em toda resposta válida.
_leakage_src2 = _leakage_path.read_text(encoding="utf-8")
_leakage_fixed2 = _leakage_src2.replace(
    '    if "No Data Leakage" in response_text:\n'
    '        leakage_status = "No Data Leakage"\n'
    '    try:',
    '    leakage_status = "No Data Leakage"  # patch (4): default antes do try\n'
    '    try:',
)
if _leakage_fixed2 != _leakage_src2:
    _leakage_path.write_text(_leakage_fixed2, encoding="utf-8")
    print("Patch upstream (4): leakage_status com default em update_extract_status")
assert '# patch (4): default antes do try' in _leakage_path.read_text(
    encoding="utf-8"
), "patch (4) não aplicou — commit do adk-samples mudou?"

# O __init__.py do sample atual importa o agente imediatamente (e tenta configurar Google
# Cloud). Criamos somente o package namespace para poder alterar CONFIG antes do grafo.
for module_name in tuple(sys.modules):
    if module_name == "machine_learning_engineering" or module_name.startswith(
        "machine_learning_engineering."
    ):
        del sys.modules[module_name]
package = py_types.ModuleType("machine_learning_engineering")
package.__file__ = str(PACKAGE_DIR / "__init__.py")
package.__package__ = "machine_learning_engineering"
package.__path__ = [str(PACKAGE_DIR)]
package.__spec__ = importlib.machinery.ModuleSpec(
    "machine_learning_engineering", loader=None, is_package=True
)
package.__spec__.submodule_search_locations = package.__path__
sys.modules["machine_learning_engineering"] = package

import litellm
from google.adk.models.lite_llm import LiteLlm
from litellm import acompletion, completion

litellm.suppress_debug_info = True  # silencia o banner "Give Feedback / Get Help" por retry


SEARCH_FAILURES = 0


async def openrouter_web_search(query: str) -> dict:
    """Search the public web through OpenRouter and return grounded evidence.

    Nunca propaga exceção: o google_search nativo do Gemini não derruba o grafo
    quando o grounding falha, então o substituto também não pode. Uma busca
    indisponível vira resultado degradado, contabilizado em SEARCH_FAILURES.
    """
    global SEARCH_FAILURES
    try:
        response = await acompletion(
            model=SEARCH_MODEL,
            api_key=openrouter_key,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Act as a web-search tool for an ML engineering agent. Search current "
                        "public sources and return concise evidence, source URLs, and directly "
                        "useful implementation details."
                    ),
                },
                {"role": "user", "content": query},
            ],
            tools=[
                {
                    "type": SEARCH_TOOL,
                    "parameters": {"engine": "native", "max_total_results": 10},
                }
            ],
            temperature=0.0,
            max_tokens=4096,
            timeout=180,
            num_retries=2,
        )
        message = response.choices[0].message
        payload = message.model_dump() if hasattr(message, "model_dump") else {}
        provider_fields = payload.get("provider_specific_fields") or {}
        annotations = payload.get("annotations") or provider_fields.get("annotations") or []
        return {"query": query, "result": message.content or "", "annotations": annotations}
    except Exception as exc:
        SEARCH_FAILURES += 1
        print(
            f"    [web_search] falha {SEARCH_FAILURES}: {type(exc).__name__}: {str(exc)[:200]}"
        )
        return {
            "query": query,
            "result": (
                f"WEB SEARCH UNAVAILABLE ({type(exc).__name__}). "
                "Proceed using your own knowledge; do not retry the search for this step."
            ),
            "annotations": [],
            "error": f"{type(exc).__name__}: {str(exc)[:300]}",
        }


# O GoogleSearchTool é específico do backend Gemini. O LiteLlm descartaria esse built-in;
# a callable abaixo vira uma FunctionTool normal e preserva o estágio Search-First.
google_search_module = importlib.import_module("google.adk.tools.google_search_tool")
google_search_module.google_search = openrouter_web_search

routed_model = LiteLlm(
    model=LITELLM_MODEL,
    api_key=openrouter_key,
    drop_params=True,
    timeout=300,
    num_retries=3,
)
# O root lê esta variável durante a construção; ele é sobrescrito pelo objeto acima logo
# após o import. Usar o slug Gemini aqui evita resolução prematura de provider.
os.environ["ROOT_AGENT_MODEL"] = "gemini-3-flash-preview"

from machine_learning_engineering.shared_libraries import config as mle_config

TASKS_DIR = SAMPLE_DIR / "machine_learning_engineering" / "tasks"

mle_config.CONFIG.agent_model = routed_model
mle_config.CONFIG.num_solutions = NUM_SOLUTIONS
mle_config.CONFIG.num_model_candidates = NUM_MODEL_CANDIDATES
mle_config.CONFIG.outer_loop_round = OUTER_LOOP_ROUND
mle_config.CONFIG.inner_loop_round = INNER_LOOP_ROUND
mle_config.CONFIG.ensemble_loop_round = ENSEMBLE_LOOP_ROUND
mle_config.CONFIG.use_data_leakage_checker = USE_DATA_LEAKAGE_CHECKER
mle_config.CONFIG.use_data_usage_checker = USE_DATA_USAGE_CHECKER
mle_config.CONFIG.exec_timeout = EXEC_TIMEOUT_S
mle_config.CONFIG.data_dir = str(TASKS_DIR) + "/"

# Só agora o import do agente constrói o grafo com OpenRouter em todos os subagentes.
from machine_learning_engineering.agent import root_agent  # noqa: E402

root_agent.model = routed_model


def walk_agents(agent):
    yield agent
    for child in getattr(agent, "sub_agents", None) or []:
        yield from walk_agents(child)


llm_agents = [agent for agent in walk_agents(root_agent) if hasattr(agent, "model")]
misrouted = [agent.name for agent in llm_agents if agent.model is not routed_model]
assert not misrouted, f"Agentes fora do OpenRouter: {misrouted}"

from machine_learning_engineering.shared_libraries import debug_util  # noqa: E402
from machine_learning_engineering.shared_libraries import code_util  # noqa: E402
from machine_learning_engineering.sub_agents.initialization import agent as init_agent  # noqa: E402

assert init_agent.google_search is openrouter_web_search
assert debug_util.google_search is openrouter_web_search

SENSITIVE_CHILD_ENV = {
    "OPENROUTER_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY",
    "OPENAI_API_KEY", "GOOGLE_API_KEY", "ANTHROPIC_API_KEY",
    "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN",
    "HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN", "WANDB_API_KEY", "GITHUB_TOKEN",
}


def run_python_code_isolated(
    code_text: str, run_cwd: str, py_filepath: str, exec_timeout: int
) -> dict:
    """Execute generated code without propagating orchestrator credentials."""
    started = time.time()
    output_path = Path(run_cwd) / py_filepath
    output_path.write_text(code_text, encoding="utf-8")
    child_home = Path(run_cwd) / ".generated_home"
    child_home.mkdir(parents=True, exist_ok=True)
    child_env = os.environ.copy()
    for secret_name in SENSITIVE_CHILD_ENV:
        child_env.pop(secret_name, None)
    child_env.update(
        HOME=str(child_home),
        XDG_CONFIG_HOME=str(child_home / ".config"),
        KAGGLE_CONFIG_DIR=str(child_home / ".kaggle"),
    )
    try:
        result = subprocess.run(
            [sys.executable, py_filepath],
            check=False, cwd=run_cwd, capture_output=True, text=True,
            timeout=exec_timeout, env=child_env,
        )
        returncode, stdout, stderr = result.returncode, result.stdout, result.stderr
    except Exception as exc:
        returncode, stdout, stderr = 1, "", str(exc)
    return {
        "returncode": returncode,
        "stdout": stdout,
        "stderr": stderr,
        "execution_time": time.time() - started,
    }


code_util.run_python_code = run_python_code_isolated

if RUN_PROVIDER_SMOKE_TEST:
    probe = completion(
        model=LITELLM_MODEL,
        api_key=openrouter_key,
        messages=[{"role": "user", "content": "Reply only with: ok"}],
        temperature=0.0,
        max_tokens=16,
        timeout=60,
        num_retries=1,
    )
    assert probe.choices[0].message.content, "OpenRouter respondeu sem conteúdo"
    search_probe = await openrouter_web_search(
        "Use web search and return the official scikit-learn homepage URL."
    )
    assert search_probe["result"] and not search_probe.get("error"), (
        f"A busca do OpenRouter falhou: {search_probe.get('error') or 'sem conteúdo'}"
    )

sample_commit = subprocess.run(
    ["git", "-C", str(SAMPLE_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert sample_commit == ADK_SAMPLES_COMMIT, (sample_commit, ADK_SAMPLES_COMMIT)
try:
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        check=False, capture_output=True, text=True,
    ).stdout.strip()
except FileNotFoundError:
    gpu_info = ""
runtime_manifest = {
    "provider": "openrouter",
    "model": MODEL,
    "litellm_model": LITELLM_MODEL,
    "search_model": SEARCH_MODEL,
    "search_tool": SEARCH_TOOL,
    "run_fingerprint": RUN_FINGERPRINT,
    "adk_samples_commit": sample_commit,
    "mlebench_commit": MLEBENCH_COMMIT,
    "google_adk_version": importlib.metadata.version("google-adk"),
    "litellm_version": importlib.metadata.version("litellm"),
    "python": sys.version,
    "platform": platform.platform(),
    "gpu": gpu_info or None,
    "generated_code_environment": "credentials_scrubbed_and_isolated_home",
    "upstream_patches": [
        "check_leakage_util.replace_leakage_code: functools.partial(prefix=prefix)",
        "common_util.get_text_from_response: Part.text None-safe",
        "agent.save_state: json.dump default=str (estado contém LiteLlm)",
        "check_leakage_util.update_extract_status: leakage_status com default",
    ],
    "protocol": {
        "seeds": SEEDS,
        "num_solutions": NUM_SOLUTIONS,
        "num_model_candidates": NUM_MODEL_CANDIDATES,
        "outer_loop_round": OUTER_LOOP_ROUND,
        "inner_loop_round": INNER_LOOP_ROUND,
        "ensemble_loop_round": ENSEMBLE_LOOP_ROUND,
        "use_data_leakage_checker": USE_DATA_LEAKAGE_CHECKER,
        "use_data_usage_checker": USE_DATA_USAGE_CHECKER,
        "max_wall_clock_s": MAX_WALL_CLOCK_S,
        "exec_timeout_s": EXEC_TIMEOUT_S,
        "max_llm_calls": MAX_LLM_CALLS,
        "skip_large_gb": SKIP_LARGE_GB,
    },
}
(RESULTS_DIR / "runtime_manifest.json").write_text(
    json.dumps(runtime_manifest, indent=2), encoding="utf-8"
)
freeze_process = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=False, capture_output=True, text=True,
)
if freeze_process.returncode == 0:
    pip_freeze = freeze_process.stdout
else:
    pip_freeze = "\n".join(sorted(
        f"{dist.metadata.get('Name', 'unknown')}=={dist.version}"
        for dist in importlib.metadata.distributions()
    )) + "\n"
(RESULTS_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")

print(f"Provider OK: {routed_model.model} | {len(llm_agents)} agentes LLM")
print("Busca Search-First OK:", SEARCH_MODEL)
print("Soluções paralelas:", mle_config.CONFIG.num_solutions)

In [ ]:
# 5) Helpers MLE-bench: preparar dados, montar task dir, grade
import json
import shutil
import subprocess


def ensure_prepared(comp_id: str) -> Path:
    """Roda `mlebench prepare -c <comp>` se necessário e retorna o dir public/."""
    public = MLEBENCH_DATA_DIR / comp_id / "prepared" / "public"
    if public.exists() and any(public.iterdir()):
        return public
    print(f"  mlebench prepare -c {comp_id} (pode demorar: download do Kaggle)")
    prepare_env = os.environ.copy()
    prepare_env.update(KAGGLE_USERNAME=kaggle_user, KAGGLE_KEY=kaggle_key)
    proc = subprocess.run(
        ["mlebench", "prepare", "-c", comp_id],
        capture_output=True, text=True, timeout=4 * 3600, env=prepare_env,
    )
    if not public.exists():
        raise RuntimeError(
            f"prepare falhou para {comp_id}:\n{proc.stdout[-1000:]}\n{proc.stderr[-1000:]}"
        )
    return public


def get_description(comp_id: str) -> str:
    """Descrição oficial da competição via registry do mlebench (com fallback)."""
    try:
        from mlebench.registry import registry

        comp = registry.set_data_dir(MLEBENCH_DATA_DIR).get_competition(comp_id)
        desc = getattr(comp, "description", "") or ""
        if desc:
            return desc
    except Exception as e:
        print(f"  registry description indisponível ({e}); usando fallback")
    for md in (MLEBENCH_DATA_DIR / comp_id).rglob("description*.md"):
        return md.read_text(encoding="utf-8", errors="ignore")
    return ""


def build_task_dir(comp: dict) -> Path:
    """Monta tasks/<comp>/ no layout do MLE-STAR: task_description.txt + dados públicos.

    O create_workspace do MLE-STAR copia TODO arquivo do task dir (exceto nomes contendo
    'answer'), então colocamos apenas o public/ do mlebench — o private/ (gabarito) fica fora.
    """
    comp_id = comp["id"]
    public = ensure_prepared(comp_id)
    task_dir = TASKS_DIR / comp_id
    if task_dir.exists():
        shutil.rmtree(task_dir)
    shutil.copytree(public, task_dir)

    files_listing = "\n".join(sorted(p.name for p in task_dir.iterdir()))
    description = get_description(comp_id)
    task_description = (
        f"# Task ({comp['task_type']})\n\n"
        f"{description}\n\n"
        f"# Metric\n{comp['metric']}\n\n"
        f"# Submission\nProduce a submission file exactly in the format of "
        f"sample_submission.csv provided with the data.\n\n"
        f"# Available data files\n{files_listing}\n"
    )
    (task_dir / "task_description.txt").write_text(task_description, encoding="utf-8")
    return task_dir


def grade_submission(comp_id: str, submission_path: Path) -> dict:
    """`mlebench grade-sample` -> dict com valid_submission/score/medals/above_median."""
    try:
        proc = subprocess.run(
            ["mlebench", "grade-sample", str(submission_path), comp_id],
            capture_output=True, text=True, timeout=300,
        )
        output = proc.stdout + proc.stderr
        start, end = output.find("{"), output.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(output[start:end])
        return {"valid_submission": False, "error": f"parse: {output[-400:]}"}
    except Exception as e:
        return {"valid_submission": False, "error": str(e)}


def cleanup_run(comp_id: str, workspace_dir: Path) -> None:
    """Remove artefatos volumosos de uma seed, preservando o download preparado."""
    for path in (TASKS_DIR / comp_id, workspace_dir):
        shutil.rmtree(path, ignore_errors=True)


def cleanup_data_cache(comp_id: str) -> None:
    """Remove os dados somente depois de concluir todas as seeds da competição."""
    shutil.rmtree(MLEBENCH_DATA_DIR / comp_id, ignore_errors=True)


print("Helpers prontos")

In [ ]:
# 6) Runner: executa o pipeline MLE-STAR de ponta a ponta para uma (competição, seed)
import asyncio
import time
import traceback

from google.adk.agents.run_config import RunConfig
from google.adk.runners import InMemoryRunner
from google.genai import types


def describe_exception(exc: BaseException, limit: int = 12000) -> tuple[str, str]:
    """(resumo da causa raiz, traceback completo) — desembrulha ExceptionGroup.

    O runner do ADK executa os agentes num asyncio.TaskGroup: str(e) vira só
    "unhandled errors in a TaskGroup" e esconde a exceção real.
    """
    root = exc
    while isinstance(root, BaseExceptionGroup) and root.exceptions:
        root = root.exceptions[0]
    summary = f"{type(root).__name__}: {root}"[:500]
    detail = "".join(traceback.format_exception(exc))
    return summary, detail[-limit:]


async def run_pipeline(comp_id: str) -> str:
    """Envia a instrução ao frontdoor agent e consome o stream de eventos do ADK."""
    runner = InMemoryRunner(agent=root_agent, app_name="mle-star-baseline")
    session = await runner.session_service.create_session(
        app_name=runner.app_name, user_id="baseline"
    )
    content = types.Content(
        parts=[types.Part(text=f"execute the {comp_id} task")], role="user"
    )
    last_text, n_events = "", 0
    async for event in runner.run_async(
        user_id=session.user_id,
        session_id=session.id,
        new_message=content,
        run_config=RunConfig(max_llm_calls=MAX_LLM_CALLS),
    ):
        n_events += 1
        author = getattr(event, "author", "?")
        if n_events % 10 == 0:
            print(f"    [{time.strftime('%H:%M:%S')}] {n_events} eventos (último: {author})")
        try:
            if event.content and event.content.parts and event.content.parts[0].text:
                last_text = event.content.parts[0].text
        except Exception:
            pass
    return last_text


def salvage_diagnostics(comp_id: str, seed: int, run_dir: Path) -> Path | None:
    """Preserva a evidência de uma run com erro ANTES de o cleanup apagar o workspace.

    Copia arquivos pequenos de texto (código gerado, logs, estados, submissões
    de até 2 MB) e a listagem completa do workspace para RESULTS_DIR.
    """
    if not run_dir.exists():
        return None
    debug_dir = RESULTS_DIR / f"debug_{comp_id}_seed{seed}"
    debug_dir.mkdir(parents=True, exist_ok=True)
    files = [p for p in sorted(run_dir.rglob("*")) if p.is_file()]
    (debug_dir / "workspace_listing.txt").write_text(
        "\n".join(str(p.relative_to(run_dir)) for p in files), encoding="utf-8"
    )
    keep_suffixes = {".py", ".json", ".txt", ".log", ".md", ".csv"}
    for path in files:
        rel = path.relative_to(run_dir)
        if "input" in rel.parts:  # dados da competição, não evidência
            continue
        if path.suffix.lower() not in keep_suffixes or path.stat().st_size > 2_000_000:
            continue
        dest = debug_dir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(path, dest)
    return debug_dir


async def run_one(comp: dict, seed: int) -> dict:
    comp_id = comp["id"]
    workspace_dir = WORK_ROOT / f"seed{seed}"
    run_dir = workspace_dir / comp_id
    result = {
        "competition_id": comp_id,
        "seed": seed,
        "model": MODEL,
        "provider": "openrouter",
        "adk_samples_commit": ADK_SAMPLES_COMMIT,
        "run_fingerprint": RUN_FINGERPRINT,
        "task_type": comp["task_type"],
        "size_gb": comp["size_gb"],
        "valid_submission": False,
        "score": None,
        "gold_medal": False,
        "silver_medal": False,
        "bronze_medal": False,
        "any_medal": False,
        "above_median": False,
        "execution_time": 0.0,
        "error": None,
    }
    start = time.time()
    search_failures_start = globals().get("SEARCH_FAILURES", 0)
    try:
        build_task_dir(comp)

        # Config runtime (lida pelo prepare_task a cada execução)
        mle_config.CONFIG.task_name = comp_id
        mle_config.CONFIG.task_type = comp["task_type"]
        mle_config.CONFIG.lower = comp["lower"]
        mle_config.CONFIG.seed = seed
        mle_config.CONFIG.workspace_dir = str(workspace_dir) + "/"

        try:
            await asyncio.wait_for(run_pipeline(comp_id), timeout=MAX_WALL_CLOCK_S)
        except asyncio.TimeoutError:
            result["error"] = f"wall-clock timeout ({MAX_WALL_CLOCK_S}s)"
            print(f"    TIMEOUT após {MAX_WALL_CLOCK_S}s — tentando grade parcial")
        except Exception as e:
            # Pipeline abortou: registra a causa raiz e SEGUE para o resgate da
            # submissão — o MLE-bench gradeia o que o agente tiver deixado.
            summary, detail = describe_exception(e)
            result["error"] = f"pipeline: {summary}"
            result["error_detail"] = detail
            print(f"    PIPELINE FALHOU: {summary}")
            print("    (traceback completo em error_detail) — tentando grade parcial")

        # Submissão final do MLE-STAR: workspace/<task>/ensemble/final/submission.csv
        # Fallback: a submissão intermediária mais recente das branches de solução.
        submission = run_dir / "ensemble" / "final" / "submission.csv"
        if not submission.exists():
            candidates = sorted(
                run_dir.rglob("submission.csv"),
                key=lambda p: p.stat().st_mtime,
                reverse=True,
            )
            submission = candidates[0] if candidates else None

        if submission and submission.exists():
            saved = RESULTS_DIR / f"submission_{comp_id}_seed{seed}.csv"
            shutil.copy(submission, saved)
            result["submission_source"] = str(submission.relative_to(run_dir))
            grade = grade_submission(comp_id, submission)
            result.update(
                valid_submission=bool(grade.get("valid_submission")),
                score=grade.get("score"),
                gold_medal=bool(grade.get("gold_medal")),
                silver_medal=bool(grade.get("silver_medal")),
                bronze_medal=bool(grade.get("bronze_medal")),
                above_median=bool(grade.get("above_median")),
                grading_output=grade,
            )
            result["any_medal"] = bool(
                result["gold_medal"] or result["silver_medal"] or result["bronze_medal"]
            )
        else:
            result["error"] = result["error"] or "no submission produced"

        # Preserva o estado final (código gerado, scores) para o material suplementar
        final_state = run_dir / "final_state.json"
        if final_state.exists():
            shutil.copy(final_state, RESULTS_DIR / f"final_state_{comp_id}_seed{seed}.json")
    except Exception as e:
        # Falha do harness (prepare/task dir/grade) — distinta de falha do pipeline
        summary, detail = describe_exception(e)
        result["error"] = f"harness: {summary}"
        result["error_detail"] = detail
        print(f"    HARNESS FALHOU: {summary}")
    finally:
        result["execution_time"] = time.time() - start
        result["search_failures"] = globals().get("SEARCH_FAILURES", 0) - search_failures_start
        if result.get("error"):
            try:
                debug_dir = salvage_diagnostics(comp_id, seed, run_dir)
                if debug_dir:
                    print(f"    Diagnóstico preservado em {debug_dir}")
            except Exception as salvage_exc:
                print(f"    Falha ao preservar diagnóstico: {salvage_exc}")
        if CLEANUP_AFTER_RUN:
            cleanup_run(comp_id, run_dir)
    return result


print("Runner pronto")

In [ ]:
# 7) Loop principal (retomável e agrupado por competição para reutilizar o download)
import re
import shutil

FATAL_PROVIDER_PATTERNS = (
    "key limit exceeded", "insufficient credits", "insufficient_quota",
    "budgetexceedederror", "authenticationerror", "permissiondeniederror",
    "invalid api key", "no auth credentials",
)


def is_fatal_provider_error(error_text) -> bool:
    """Erro de chave/créditos/permissão do provider: derruba toda chamada seguinte."""
    text = str(error_text or "").lower()
    if re.search(r'"code"\s*:\s*40[123]\b', text):
        return True
    return any(pattern in text for pattern in FATAL_PROVIDER_PATTERNS)


FATAL_HARNESS_PATTERNS = (
    "no such file or directory: 'mlebench'",
    "no space left on device",
    "disk quota exceeded",
    "read-only file system",
)


def is_fatal_harness_error(error_text) -> bool:
    """Ambiente quebrado (CLI ausente, disco cheio): nada depois vai funcionar."""
    text = str(error_text or "").lower()
    return any(pattern in text for pattern in FATAL_HARNESS_PATTERNS)


def is_final_result(r: dict) -> bool:
    """Resultado que NÃO deve re-rodar no resume.

    Final: pulado; wall-clock esgotado (o orçamento do protocolo, com ou sem
    submissão); pipeline terminou limpo (error=None ou "no submission
    produced" = falha genuína do sistema). Parcial (re-roda): pipeline morto
    por bug/limite de harness — mesmo que uma submissão parcial tenha sido
    resgatada e gradeada, ela subestima o baseline. Exceção: salvage com
    MEDALHA DE OURO é final — ouro é o teto da métrica, re-rodar não pode
    melhorar, e mantê-lo é conservador a favor do baseline.
    """
    if r.get("skipped") is True:
        return True
    error = str(r.get("error") or "")
    if error.startswith("wall-clock timeout"):
        return True
    if error == "no submission produced":
        return True
    # Ouro resgatado é terminal: teto da métrica; re-rodar não melhora e
    # manter favorece o baseline (prata/bronze re-rodam: podem virar ouro).
    if r.get("valid") is True and r.get("gold_medal") is True:
        return True
    return not error


def fingerprint_core(fingerprint) -> str:
    """Núcleo do fingerprint que define a compatibilidade de UM resultado.

    Ignora o roster de competições/seeds e o cap de disco: editar a lista no
    Colab não pode invalidar resultados já obtidos com o mesmo modelo, mesmos
    commits e mesmo protocolo (foi o que arquivou runs boas como
    "incompatíveis" quando o roster foi cortado de 22 para 15).
    """
    if not isinstance(fingerprint, str) or not fingerprint:
        return ""
    head = fingerprint.rsplit(":", 1)[0]  # openrouter:<modelo>:<adk>:<mlebench>
    protocol = re.search(
        r"n\d+-m\d+-o\d+-i\d+-e\d+-leak\d+-usage\d+-exec\d+-wall\d+", fingerprint
    )
    return f"{head}|{protocol.group(0) if protocol else fingerprint}"


def dedupe_results(results: list[dict]) -> list[dict]:
    """Um resultado por (competição, seed): válido > pulado > falha; empate = mais recente."""
    def rank(r: dict) -> tuple[int, int]:
        return (
            1 if is_final_result(r) else 0,
            1 if r.get("valid_submission") else 0,
        )

    best: dict[tuple, dict] = {}
    for r in results:
        key = (r.get("competition_id"), r.get("seed", 42))
        if key not in best or rank(r) >= rank(best[key]):
            best[key] = r
    return list(best.values())


def assert_provider_alive() -> None:
    """Sonda barata antes de gastar download/tempo de pipeline.

    Uma chave morta (403 key-limit, 402 créditos) queimou 13 competições em 20
    minutos; com a sonda, o loop para ANTES do `mlebench prepare` da próxima.
    """
    try:
        probe = completion(
            model=LITELLM_MODEL,
            api_key=openrouter_key,
            messages=[{"role": "user", "content": "Reply only with: ok"}],
            temperature=0.0,
            max_tokens=8,
            timeout=60,
            num_retries=1,
        )
        assert probe.choices[0].message.content, "resposta vazia"
    except Exception as exc:
        raise RuntimeError(
            "Preflight do OpenRouter falhou — loop pausado antes de queimar as "
            "competições. Corrija a chave/créditos em openrouter.ai e re-execute "
            f"esta célula (o resume continua de onde parou). Causa: {exc}"
        ) from exc


def assert_environment_ready() -> None:
    """Pré-checagem do harness antes do loop (par do preflight de provider)."""
    if shutil.which("mlebench") is None:
        raise RuntimeError(
            "CLI `mlebench` não encontrado — a célula 1 (instalação) não rodou nesta "
            "sessão ou o pip dela falhou. Re-execute a célula 1, confira o print "
            "'Instalação OK' e volte aqui (os pares com erro re-rodam sozinhos)."
        )


RUN_CORE = fingerprint_core(RUN_FINGERPRINT)
results_path = RESULTS_DIR / "results_mlestar.json"
main_loaded = json.loads(results_path.read_text()) if results_path.exists() else []
incompatible_results = [
    r for r in main_loaded if fingerprint_core(r.get("run_fingerprint")) != RUN_CORE
]
if incompatible_results:
    archive_path = RESULTS_DIR / (
        f"results_mlestar_incompatible_{time.strftime('%Y%m%d-%H%M%S')}.json"
    )
    archive_path.write_text(json.dumps(incompatible_results, indent=2, default=str))
    print(f"Resultados de outra configuração preservados em {archive_path}")
compatible_main = [
    r for r in main_loaded if fingerprint_core(r.get("run_fingerprint")) == RUN_CORE
]

# Readota dos arquivos "incompatíveis" antigos tudo que tem o mesmo núcleo de config
# (ex.: resultados válidos arquivados só porque o roster foi editado).
rescued = []
for archive in sorted(RESULTS_DIR.glob("results_mlestar_incompatible_*.json")):
    try:
        rescued += [
            r
            for r in json.loads(archive.read_text())
            if fingerprint_core(r.get("run_fingerprint")) == RUN_CORE
        ]
    except Exception as exc:
        print(f"Ignorando arquivo ilegível {archive.name}: {exc}")
main_keys = {(r.get("competition_id"), r.get("seed", 42)) for r in compatible_main}
n_rescued = sum(
    1 for r in rescued if (r.get("competition_id"), r.get("seed", 42)) not in main_keys
)
if n_rescued:
    print(f"{n_rescued} resultado(s) readotado(s) de arquivos anteriores (mesmo núcleo de config)")
all_results = dedupe_results(rescued + compatible_main)

if FORCE_RERUN and all_results:
    rerun_archive = RESULTS_DIR / (
        f"results_mlestar_forced_rerun_{time.strftime('%Y%m%d-%H%M%S')}.json"
    )
    rerun_archive.write_text(json.dumps(all_results, indent=2, default=str))
    print(f"Resultados anteriores preservados em {rerun_archive}")
    all_results = []
done = {
    (r["competition_id"], r.get("seed", 42))
    for r in all_results
    if not FORCE_RERUN and is_final_result(r)
}

assert_environment_ready()
assert_provider_alive()
quick_failures = 0
for i, comp in enumerate(COMPETITIONS, 1):
    oversized = SKIP_LARGE_GB is not None and comp["size_gb"] > SKIP_LARGE_GB
    for seed in SEEDS:
        key = (comp["id"], seed)
        header = f"[{i}/{len(COMPETITIONS)}] {comp['id']} (seed={seed})"
        if key in done:
            print(f"{header} — já executado, pulando")
            continue
        all_results = [
            previous for previous in all_results
            if (previous["competition_id"], previous.get("seed", 42)) != key
        ]
        if oversized:
            print(f"{header} — PULADO ({comp['size_gb']} GB > {SKIP_LARGE_GB} GB)")
            all_results.append({
                "competition_id": comp["id"], "seed": seed, "model": MODEL,
                "provider": "openrouter", "adk_samples_commit": ADK_SAMPLES_COMMIT,
                "run_fingerprint": RUN_FINGERPRINT,
                "task_type": comp["task_type"], "size_gb": comp["size_gb"],
                "skipped": True, "error": f"skipped: size > {SKIP_LARGE_GB} GB",
                "valid_submission": False, "any_medal": False, "above_median": False,
            })
            results_path.write_text(json.dumps(all_results, indent=2, default=str))
            continue
        print(f"\n{'=' * 70}\n{header}\n{'=' * 70}")
        assert_provider_alive()
        result = await run_one(comp, seed)
        result["infra_error"] = is_fatal_provider_error(
            result.get("error")
        ) or is_fatal_harness_error(result.get("error"))
        all_results.append(result)
        results_path.write_text(json.dumps(all_results, indent=2, default=str))
        medal = "🥇" if result["gold_medal"] else "🥈" if result["silver_medal"] else "🥉" if result["bronze_medal"] else "—"
        print(
            f"  -> valid={result['valid_submission']} score={result['score']} "
            f"medal={medal} above_median={result['above_median']} "
            f"({result['execution_time'] / 3600:.2f}h) erro={result['error']}"
        )
        if result["infra_error"]:
            raise RuntimeError(
                "Erro fatal de provider/ambiente no meio da run — loop interrompido "
                "para não registrar falhas espúrias nas demais competições. Corrija e "
                "re-execute esta célula (este par re-roda sozinho). Causa: "
                f"{str(result['error'])[:300]}"
            )
        if result.get("error") and result["execution_time"] < 300:
            quick_failures += 1
        else:
            quick_failures = 0
        if quick_failures >= 3:
            raise RuntimeError(
                "3 falhas rápidas consecutivas (<5 min) — provável problema sistêmico "
                "não catalogado (ambiente/harness). Veja error_detail no "
                "results_mlestar.json e os diretórios debug_*/; os pares com erro "
                "re-rodam sozinhos quando você re-executar esta célula."
            )
    if CLEANUP_AFTER_RUN:
        cleanup_data_cache(comp["id"])

print(f"\nConcluído. Resultados em {results_path}")

In [ ]:
# 8) Validar completude antes de publicar a tabela final
from collections import Counter

final_results = json.loads(results_path.read_text())

infra_failures = [r for r in final_results if r.get("infra_error")]
assert not infra_failures, (
    f"{len(infra_failures)} resultado(s) com erro fatal de provider "
    f"(ex.: {infra_failures[0]['competition_id']}: {str(infra_failures[0].get('error'))[:120]}). "
    "Corrija a chave/créditos e re-execute a célula 7 — esses pares re-rodam sozinhos."
)

partial_results = [r for r in final_results if not is_final_result(r)]
assert not partial_results, (
    f"{len(partial_results)} resultado(s) parciais (pipeline abortado por harness), "
    f"ex.: {partial_results[0]['competition_id']}: "
    f"{str(partial_results[0].get('error'))[:120]}. "
    "Re-execute a célula 7 — eles re-rodam sozinhos."
)

result_keys = [(r["competition_id"], r.get("seed", 42)) for r in final_results]
expected_keys = {(comp["id"], seed) for comp in COMPETITIONS for seed in SEEDS}
duplicate_keys = [key for key, count in Counter(result_keys).items() if count > 1]
missing_keys = sorted(expected_keys - set(result_keys))
extra_keys = sorted(set(result_keys) - expected_keys)
assert not duplicate_keys, f"Resultados duplicados: {duplicate_keys}"
if extra_keys:
    # Resultados de um roster anterior (lista editada): válidos, só não estão na lista atual.
    print(f"Aviso: {len(extra_keys)} par(es) fora do roster atual mantidos no JSON: {extra_keys[:5]}")
assert not missing_keys, (
    f"Execução parcial: faltam {len(missing_keys)} pares competição×seed. "
    "Retome a célula 7 antes de gerar a tabela final."
)
print(f"Completude OK: {len(expected_keys)} pares do roster atual presentes, sem duplicatas")

In [ ]:
# 9) Agregação + tabela comparativa (formato da Table 5 da monografia)
import pandas as pd

df = pd.DataFrame(json.loads(results_path.read_text()))
skipped_mask = df.get("skipped", pd.Series(False, index=df.index)).fillna(False).astype(bool)
infra_mask = df.get("infra_error", pd.Series(False, index=df.index)).fillna(False).astype(bool)
attempted = df[~skipped_mask & ~infra_mask]
n = len(attempted)
if int(infra_mask.sum()):
    print(
        f"Aviso: {int(infra_mask.sum())} run(s) com erro de infra excluída(s) da tabela — "
        "re-execute a célula 7 para completá-las"
    )


def pct(series) -> float:
    return 100.0 * series.fillna(False).astype(bool).sum() / max(n, 1)


mlestar_row = {
    "Sistema": f"MLE-STAR ({MODEL}, este notebook, n={n})",
    "Submissão válida %": round(pct(attempted["valid_submission"]), 1),
    "Acima da mediana %": round(pct(attempted["above_median"]), 1),
    "Medalhas %": round(pct(attempted["any_medal"]), 1),
    "Ouro %": round(pct(attempted.get("gold_medal", pd.Series(dtype=bool))), 1),
}

# Referências: tese (Kaggle Agents) e paper do MLE-STAR (Nam et al., 2025)
reference_rows = [
    {"Sistema": "AIDE (Gemini 2.0 Flash, paper)",        "Submissão válida %": 78.8,  "Acima da mediana %": 39.4, "Medalhas %": 25.8, "Ouro %": 12.1},
    {"Sistema": "MLE-STAR (Gemini 2.0 Flash, paper)",    "Submissão válida %": 95.5,  "Acima da mediana %": 63.6, "Medalhas %": 43.9, "Ouro %": 30.3},
    {"Sistema": "MLE-STAR (Gemini 2.5 Pro, paper)",      "Submissão válida %": 100.0, "Acima da mediana %": 83.3, "Medalhas %": 63.6, "Ouro %": 36.4},
    {"Sistema": "Kaggle Agents (Gemini 3.0 Flash, tese, protocolo legado)", "Submissão válida %": 100.0, "Acima da mediana %": 72.7, "Medalhas %": 59.1, "Ouro %": 27.3},
]

comparison = pd.DataFrame(reference_rows + [mlestar_row])
display(comparison)

print("\nPor competição:")
cols = ["competition_id", "seed", "valid_submission", "score", "any_medal", "above_median", "submission_source", "search_failures", "execution_time", "error"]
display(df[[c for c in cols if c in df.columns]])

total_h = attempted["execution_time"].fillna(0).sum() / 3600 if "execution_time" in attempted.columns else 0.0
print(f"\nTempo total de execução: {total_h:.1f} h GPU (custo L4 ~US$ {total_h * 0.67:.0f})")
comparison.to_csv(RESULTS_DIR / "comparison_table.csv", index=False)

In [ ]:
# 10) Dispersão entre seeds (não agregamos scores brutos de métricas diferentes)
seed_stats = (
    attempted.groupby("seed")
    .agg(
        n=("competition_id", "size"),
        valid_submission_rate=("valid_submission", "mean"),
        above_median_rate=("above_median", "mean"),
        any_medal_rate=("any_medal", "mean"),
        execution_time_h=("execution_time", lambda values: values.sum() / 3600),
    )
)
rate_columns = ["valid_submission_rate", "above_median_rate", "any_medal_rate"]
seed_stats[rate_columns] = 100 * seed_stats[rate_columns]
display(seed_stats.round(2))
display(seed_stats[rate_columns].agg(["mean", "std"]).round(2))

## Notas metodológicas (para a seção de experimentos do paper)

1. **Backbone pareado:** este baseline usa Gemini 3.0 Flash Preview
   (`google/gemini-3-flash-preview`) via OpenRouter, o mesmo backbone do Kaggle Agents,
   eliminando o confound de geração de modelo da comparação original (paper: 2.0 Flash / 2.5 Pro).
2. **Mesma infra:** uma GPU de Colab, mesmo disco/CPU — o custo por competição passa a ser *medido*
   na mesma máquina, e não estimado por proxy de preço de V100.
3. **Seeds:** este notebook já usa `SEEDS = [42, 43, 44]`, como o protocolo de 3 seeds do
   MLE-STAR; reporte média, dispersão e teste pareado por competição.
4. **Loops:** `NUM_MODEL_CANDIDATES=4` e os loops 4/4/5 já estão configurados para a
   paridade planejada. Registre qualquer redução de orçamento antes de comparar resultados.
5. **Busca/contaminação:** o `google_search` nativo do ADK não é transportado pelo LiteLLM.
   O notebook o substitui por uma FunctionTool que chama o server tool atual
   `openrouter:web_search` com o mesmo Gemini 3.0, preservando o estágio Search-First, mas não
   uma política de recuperação
   idêntica à do paper. Ela continua sem filtro anti-contaminação por competição; o Kaggle Agents
   aplica esse filtro. Reporte explicitamente essa diferença metodológica.
6. **Reprodutibilidade:** os commits do `adk-samples` e MLE-bench, Google ADK e LiteLLM estão
   fixados; `runtime_manifest.json` e `pip_freeze.txt` registram software e hardware. Resultados
   persistem no Drive e a retomada aceita apenas submissões válidas do mesmo fingerprint.
7. **Competições puladas por disco** (`SKIP_LARGE_GB`) devem ser reportadas explicitamente na
   tabela (ex.: siim-isic ~116 GB não cabe no disco do Colab porque o MLE-STAR copia os dados
   por branch de solução). O default de 20 GB a omite; use `None` apenas com disco suficiente.
8. **Limite de 24h:** `asyncio.wait_for` é um limite de orquestração, não um hard kill para
   ferramentas síncronas; registre o wall-clock real e qualquer excesso de até um timeout de componente.
9. **Política de falhas e resgate:** o runner desembrulha o `ExceptionGroup` do ADK e grava a
   causa raiz (`error`) + traceback completo (`error_detail`); mesmo com crash do pipeline, a
   submissão mais recente do workspace é gradeada (`submission_source` distingue
   `ensemble/final/submission.csv` de artefato parcial) e a evidência é copiada para
   `debug_<comp>_seed<seed>/` antes do cleanup. Falhas do adapter de busca não derrubam o
   grafo (contadas por run em `search_failures`). Para a tabela final: erro de infra/adapter →
   re-executar o par competição×seed (o resume faz isso sozinho, pois só `valid=True`/`skipped`
   entram no `done`); conta contra o sistema apenas falha do próprio agente dentro do orçamento
   (sem submissão gradeável até o limite de 24h).
10. **Parada em erro fatal de provider:** um preflight barato roda antes de cada par
   competição×seed (uma chave com "Key limit exceeded" queimou 13 competições em ~20 min,
   inclusive downloads do `mlebench prepare`); o primeiro resultado com `infra_error=True`
   interrompe o loop, e validação/agregação excluem essas entradas da tabela. Resultados de
   rosters/seeds anteriores são readotados por `fingerprint_core` (mesmo modelo, commits e
   protocolo — a lista de competições e o cap de disco não invalidam resultados individuais).
11. **Orçamento de chamadas e patches upstream:** o `RunConfig.max_llm_calls` default do ADK
   (500) truncava as runs em ~4h (`LlmCallsLimitExceededError`); o notebook agora roda com
   `MAX_LLM_CALLS=4000` como guarda anti-runaway — o orçamento do protocolo continua sendo o
   wall-clock de 7h (o teto não entra no fingerprint; runs que o atingirem ficam com `error`
   registrado e re-rodam). Quatro bugs do sample oficial são corrigidos no clone antes do
   import (listados em `runtime_manifest.json`): o callback `replace_leakage_code` sem
   `prefix` (só alcançável com o leakage checker LIGADO, como no paper) e a concatenação de
   `Part.text=None` em `get_text_from_response` e o `json.dump` do estado final em
   `save_state` sem `default=str` (o estado carrega o objeto LiteLlm da troca de
   backbone e derrubava toda run completada) e o `leakage_status` sem default em
   `update_extract_status` (resposta vazia do provedor → UnboundLocalError no fim do
   callback). Resultados **parciais** (pipeline morto por
   harness com submissão resgatada) nunca entram na tabela nem no `done`: `is_final_result`
   governa resume e validação — com uma exceção: salvage que já obteve **medalha de
   ouro** (teto da métrica) é terminal, pois re-rodar não pode melhorar o resultado;
   manter o ouro de uma run truncada é conservador a favor do baseline (prata/bronze
   re-rodam, pois ainda podem virar ouro). Divulgado na seção de experimentos.
12. **Pré-checagem de ambiente e circuit breaker:** além do preflight de provider, a célula 7
   valida o CLI `mlebench` antes do loop (uma sessão sem a célula 1 executada queimou 12
   competições em segundos com `FileNotFoundError: 'mlebench'`); erros fatais de harness (CLI
   ausente, disco cheio) param o loop como os de provider, e 3 falhas rápidas consecutivas
   (<5 min) interrompem por suspeita de problema sistêmico não catalogado. Entradas com erro
   nunca são finais (`is_final_result`) e re-rodam no resume. A instalação do mle-bench usa
   o tarball do commit (codeload) com retries e fallback para git: o `git clone` anônimo
   sofre throttling intermitente no Colab (exit 128), mesmo com o repo público e íntegro.